# Stage A on Kaggle — extract your shard of the feature bank

**You are one of five.** Each of us extracts a contiguous fifth of the same
frozen manifest into a *shard bank*; one person merges the five afterwards.
Because every view's pixels depend only on `(seed, row_id, view_idx)`, five
shards recombine into a bank that is **bit-identical** to one extracted in a
single uninterrupted run — provided nobody changes the parameters below except
`SHARD_INDEX`.

## Before you start

1. **Settings → Accelerator → GPU T4 ×2** (or P100). Without this, extraction
   runs on CPU and will not finish.
2. **Settings → Internet → On** (needed for the clone and the model download).
3. **Add data**: attach the `techjam-aigc-train` Dataset (it carries the images
   **and** `manifest.parquet`). Ask the owner to share it with you first.
4. **Add-ons → Secrets**: a secret named `HF_TOKEN` holding *your own*
   HuggingFace read token. See the auth step below — the DINOv3 licence is
   accepted **per account**.
5. Agree your `SHARD_INDEX` with the team **before** running. Two people on the
   same index waste one of the two sessions: `merge_banks` refuses overlapping
   shards.

## Read this once — the four ways to lose a day

| Do not | Because |
| --- | --- |
| Do not run `scripts/build_dataset.py` | The manifest is **frozen**. Rebuilding it re-splits the data, and re-splitting after banks exist silently misaligns labels against cached features. Nothing errors; the numbers just get quietly worse. |
| Do not `pip install torch` (or `-e .` without `--no-deps`) | Kaggle's torch is matched to its drivers. Replacing it costs you the GPU for the rest of the session. The install cell below is built to avoid this. |
| Do not narrow `SPLITS` to `train` | Stage B evaluates on the bank's own `val_internal` rows. A train-only bank is unusable and the 8–13 h has to be paid again. |
| Do not write the bank to `/kaggle/temp` | It is discarded when the session ends — which is exactly the event `--resume` exists to survive. |

If a cell fails, **do not just re-run it**. Jump to *The 2am playbook* at the
bottom: it tells you whether a re-run can possibly help.

## 0. Parameters

`SHARD_INDEX` is the only line most people change. `SMOKE = True` proves the
whole chain on a handful of images in a couple of minutes; leave it on for the
first run of the day, then set it to `False` for the real thing.

In [25]:
# ============ THE ONLY LINES YOU NORMALLY EDIT ============
SHARD_INDEX = 1          # YOURS: 0, 1, 2, 3 or 4. Agree it with the team first.
N_SHARDS    = 5          # one per teammate. Everyone must use the SAME number.
SMOKE       = False       # True = prove the chain in minutes; False = the real run
# ==========================================================

BACKBONE = "siglip2l"      # the FLEET's backbone; dinov3l runs on the A4500
SPLITS   = "train,val_internal"   # Stage B needs BOTH. Do not narrow this.
SEED     = 20260827               # must match scripts/extract_features.py

# Tuning. Lower BATCH_SIZE on an OOM; lower WORKERS if the kernel runs out of RAM.
WORKERS          = 4
BATCH_SIZE       = 16
CHECKPOINT_EVERY = 200            # images between metadata flushes = work at risk

REPO_URL = "https://github.com/bersamin12/robust-aigc-detection"
BRANCH   = "feat/robust-aigc-detection"
REPO_DIR = "/kaggle/working/robust-aigc-detection"

# Where the attached Datasets landed. The glob is deliberate: the normalised
# dataset is published as several Kaggle Datasets because of the per-Dataset
# size cap, and they mount at separate paths.
MANIFEST_GLOB = "/kaggle/input/datasets/justinbersamin/techjam-aigc-train*/manifest.parquet"
DATA_GLOB     = "/kaggle/input/datasets/justinbersamin/techjam-aigc-train*"

# The bank goes in /kaggle/working: it is the only location that survives the
# session as notebook output, and surviving is the whole point of --resume.
OUT_DIR = f"/kaggle/working/banks/{BACKBONE}_shard{SHARD_INDEX}"

# Derived, never typed: the merge notebook globs for exactly this name, and
# a hand-written "dinov3l" in a siglip2l run produces a Dataset the merge
# silently cannot find.
BANK_DATASET = f"aigcdet-bank-{BACKBONE}-shard{SHARD_INDEX}"

print(f"shard {SHARD_INDEX} of {N_SHARDS}  backbone={BACKBONE}  "
      f"smoke={SMOKE}\n  -> {OUT_DIR}\n  publish as: {BANK_DATASET}")

shard 1 of 5  backbone=siglip2l  smoke=False
  -> /kaggle/working/banks/siglip2l_shard1
  publish as: aigcdet-bank-siglip2l-shard1


## 1. Get the code

Public repo, shallow clone, and a `reset --hard` on re-run so a resumed session
picks up any fix without a stale working tree. Nothing here is authenticated.

In [26]:
import glob, importlib, os, subprocess, sys, time

def sh(argv, **kw):
    """Run a command, show it, and fail loudly rather than continuing."""
    print("$", " ".join(str(a) for a in argv))
    return subprocess.run([str(a) for a in argv], check=True, **kw)

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH])
    sh(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])
else:
    sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])

if os.path.join(REPO_DIR, "notebooks") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "notebooks"))
import kaggle_bootstrap as kb
importlib.reload(kb)

sh(["git", "-C", REPO_DIR, "log", "--oneline", "-1"])
print("helper loaded from", kb.__file__)

$ git -C /kaggle/working/robust-aigc-detection fetch --depth 1 origin feat/robust-aigc-detection
$ git -C /kaggle/working/robust-aigc-detection reset --hard origin/feat/robust-aigc-detection
HEAD is now at 58b8a6f feat(predict): the deliverable emits the calibrated probability
$ git -C /kaggle/working/robust-aigc-detection log --oneline -1
58b8a6f feat(predict): the deliverable emits the calibrated probability
helper loaded from /kaggle/working/robust-aigc-detection/notebooks/kaggle_bootstrap.py


From https://github.com/bersamin12/robust-aigc-detection
 * branch            feat/robust-aigc-detection -> FETCH_HEAD
 + 5612500...58b8a6f feat/robust-aigc-detection -> origin/feat/robust-aigc-detection  (forced update)


## 2. Install — without losing Kaggle's torch

This is the step that ends sessions. `pip install -e .` hands pip the
`torch>=2.0` line from `pyproject.toml` and invites it to resolve a torch built
for a different CUDA than this machine's drivers; you get a `torch` that cannot
see the GPU and no way back except a factory reset.

So: the project goes in with `--no-deps` (a pure path registration, which is all
it is needed for), everything else is installed **only if genuinely missing**,
and nothing CUDA-matched is touched at all. `transformers` is the one exception
— Kaggle images move and the project needs ≥4.53 for DINOv3 — and it is
upgraded with `--no-deps` too.

**Print the plan before running it.** If you ever see `torch` in that list,
stop.

In [27]:
def installed_version(dist):
    try:
        import importlib.metadata as im
        return im.version(dist)
    except Exception:
        return None

plan = kb.install_plan(os.path.join(REPO_DIR, "pyproject.toml"), REPO_DIR,
                       transformers_version=installed_version("transformers"))

print("pip plan:")
for cmd in plan:
    print("   ", " ".join(cmd))

assert not any(w.split("=")[0].split(">")[0] in ("torch", "torchvision", "triton")
               for cmd in plan for w in cmd), "STOP: the plan would touch torch"

for cmd in plan:
    sh(cmd, capture_output=True, text=True)
print("\ninstall done")

pip plan:
    /usr/bin/python3 -m pip install --no-deps -e /kaggle/working/robust-aigc-detection
$ /usr/bin/python3 -m pip install --no-deps -e /kaggle/working/robust-aigc-detection

install done


### 2b. Check the environment before paying for anything

Every problem this cell can report makes the 1.2 GB model download pointless,
so it runs before the download rather than after it.

If it tells you `transformers` was upgraded: **restart the kernel**
(Run → Restart session) and re-run from cell 0. A module already imported at
the old version stays imported.

In [28]:
import platform

torch_v = installed_version("torch")
tf_v    = installed_version("transformers")
problems = kb.environment_problems(platform.python_version(), torch_v, tf_v)

print(f"python {platform.python_version()}  torch {torch_v}  transformers {tf_v}")
if problems:
    for p in problems:
        print("\nPROBLEM:", p)
    raise SystemExit("fix the above before continuing")

import torch
print("cuda available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), (
    "no GPU. Settings > Accelerator > GPU, then restart the session. "
    "Extraction on CPU will not finish inside a session.")

python 3.12.13  torch 2.10.0+cu128  transformers 5.0.0
cuda available: True | Tesla T4


## 3. HuggingFace auth

**Only if your `BACKBONE` is gated.** SigLIP2 (Apache-2.0) and CLIP (MIT) are
public: the cell below will say so and move on, and you need no token at all.
DINOv3 is gated behind Meta's licence, and then two separate things must be
true — from this notebook they fail identically, as a 401/403 on
`from_pretrained`:

1. **Your own** HuggingFace account has accepted the licence at the model page.
   Acceptance is per account — the project owner's acceptance does nothing for
   yours.
2. A read token from that same account is attached to this notebook as a Kaggle
   Secret named `HF_TOKEN`.

**Never paste a token into a cell.** This repo is public and a notebook is
committed with its cell source. Add-ons → Secrets is the whole reason that
mechanism exists.

In [29]:
import aigcdet

In [30]:
# Both read off the registry, so they follow BACKBONE rather than being
# typed again here. DINOv3 is gated behind Meta's licence; SigLIP2
# (Apache-2.0) and CLIP (MIT) are not, and the fleet should not be
# stopped for a token its run never uses.

from aigcdet.features.backbones import BACKBONES
MODEL_ID = BACKBONES[BACKBONE].hf_id
GATED    = kb.requires_hf_token(BACKBONE)

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except Exception:
    secrets = None

token = kb.hf_token(secrets)
for line in kb.hf_auth_advice(token, MODEL_ID, gated=GATED):
    print(line)

if token:
    # Exported for `transformers` to pick up. Never printed, never written to a
    # file that leaves this session.
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
elif GATED:
    raise SystemExit("no HuggingFace token -- see the instructions above")

google/siglip2-large-patch16-384 is a public repo -- no token is required, and none was needed to reach this cell.
If a download fails here it is the network or the mirror, not authentication. Re-run the cell.


## 4. Attach the data, and prove it is intact — **do not skip this**

Two things happen here.

**The mounts become one tree.** The manifest describes a single dataset root,
and Kaggle mounts each published Dataset at its own `/kaggle/input/<slug>/`. A
symlink farm in `/kaggle/temp` presents them as the one tree the manifest can be
rebased onto. Symlinks, not copies: the data is tens of GB and `/kaggle/input`
is read-only. `/kaggle/temp` is the right home for it precisely because it does
*not* persist — it costs nothing to rebuild next session.

**The files are checked against the frozen manifest.** `verify_images`
recomputes each file's digest and reports which of *missing* / *unreadable* /
*content-divergent* is wrong, because the fixes differ. This costs minutes.
Skipping it costs 8–13 GPU-hours of features that do not correspond to the
manifest's labels — and nothing downstream would tell you.

You cannot skip it by accident: the cell produces `GATE`, and every later cell
takes `GATE` as a required argument. No gate, no extraction.

> During a `SMOKE` run only a sample is digested, so it is quick. The real run
> digests everything. A clean sample is evidence, not proof, and the printout
> keeps saying so.

In [31]:
import pandas as pd

MANIFEST = sorted(glob.glob(MANIFEST_GLOB))
DATA_MOUNTS = sorted(glob.glob(DATA_GLOB))
assert MANIFEST, f"no manifest Dataset attached (looked for {MANIFEST_GLOB})"
assert DATA_MOUNTS, f"no image Datasets attached (looked for {DATA_GLOB})"
MANIFEST = MANIFEST[0]
print(f"manifest: {MANIFEST}")
print(f"{len(DATA_MOUNTS)} image Dataset(s):")
for m in DATA_MOUNTS:
    print("   ", m)

# What the manifest says its root contains -- this is what locates the root
# inside each mount, instead of guessing at Kaggle's wrapper directories.
EXPECTED = kb.top_level_names(pd.read_parquet(MANIFEST, columns=["rel_path"]))
print("dataset root should contain:", sorted(EXPECTED))

UNIFIED = kb.unify_mounts(DATA_MOUNTS, "/kaggle/temp/aigcdet_root", EXPECTED)
DATA_ROOT = UNIFIED.root
print("unified root:", DATA_ROOT, "->", sorted(os.listdir(DATA_ROOT))[:8])

manifest: /kaggle/input/datasets/justinbersamin/techjam-aigc-train/manifest.parquet
1 image Dataset(s):
    /kaggle/input/datasets/justinbersamin/techjam-aigc-train
dataset root should contain: ['sid_set', 'wildfake']
unified root: /kaggle/temp/aigcdet_root -> ['manifest.parquet', 'sid_set', 'wildfake']


In [32]:
t0 = time.time()
manifest, GATE = kb.open_verified_manifest(
    MANIFEST, DATA_ROOT,
    sample=2000 if SMOKE else None,
    # os.walk does not follow the farm's symlinked directories, so an
    # "extra files: 0" from it would be unearned. Skipped and said so.
    check_extra=not UNIFIED.linked,
)
print(kb.describe_gate(GATE))
print(f"\nverified in {time.time() - t0:.0f}s")
print(manifest["split"].value_counts().to_string())

verified: 138116 rows under /kaggle/temp/aigcdet_root
  digest bytes over 138116 rows
  manifest fingerprint 7782a9ff346098d6...

verified in 240s
split
train                117784
val_internal          13332
heldout_generator      7000


## 5. Your shard

Shards are **contiguous** slices of the split-filtered manifest, and the slice
keeps the manifest's **original index labels**. Both halves matter:

* *Contiguous*, because `merge_banks` concatenates shards in the order given and
  the result is read positionally against the manifest. A strided split
  (`iloc[k::5]`) would produce rows `0,5,10,…,1,6,11,…` — a permutation that
  merges without complaint and then misreads every label.
* *Original index labels*, because `extract_bank` derives every view's RNG from
  `(seed, row_id, view_idx)` where `row_id` **is** that label. Resetting the
  index would restart each shard's keys at 0, collide all five in key space, and
  give the same physical image different pixels depending on who extracted it.
  Nothing would raise.

The table below is the assignment. Check your row against the team chat.

In [33]:
plan_rows = kb.shard_plan(GATE, manifest, N_SHARDS, splits=SPLITS)
display(pd.DataFrame(plan_rows))

mine = kb.shard_frame(GATE, manifest, SHARD_INDEX, N_SHARDS, splits=SPLITS)
print(f"\nYOURS: shard {SHARD_INDEX} -- {len(mine)} images, "
      f"row_id {int(mine.index[0])}..{int(mine.index[-1])}")

from aigcdet.features.backbones import BACKBONES
DIM = BACKBONES[BACKBONE].dim

fits, sizing = kb.fits_in_working(len(mine), DIM)
print(sizing)
assert fits, (
    "this shard's bank cannot fit /kaggle/working. The .npy files are "
    "preallocated at FULL size before the first image, so this would fail at "
    "the end with nothing saved. Raise N_SHARDS (everyone must use the same "
    "value) and re-run.")

,shard,rows,iloc,row_id_first,row_id_last,n_fake,splits
0,0,26224,[0:26224),0,26223,16175,"{'train': 23544, 'val_internal': 2680}"
1,1,26223,[26224:52447),26224,59446,26223,"{'train': 23582, 'val_internal': 2641}"
2,2,26223,[52447:78670),59447,85669,13169,"{'train': 23506, 'val_internal': 2717}"
3,3,26223,[78670:104893),85670,111892,0,"{'train': 23609, 'val_internal': 2614}"
4,4,26223,[104893:131116),111893,138115,10500,"{'train': 23543, 'val_internal': 2680}"



YOURS: shard 1 -- 26223 images, row_id 26224..59446
bank 0.57 GiB + 0.50 GiB reserve vs 20 GiB working quota: fits


## 6. Smoke run — prove the whole chain in a couple of minutes

Two tiny extractions, of different sizes, into a throwaway directory. Two,
because one measures the wrong thing: most of a 40-image run is the model
download, the CUDA context and the process start. Subtracting the small run from
the large one leaves only the cost that actually scales, which is the number the
rest of the plan depends on.

Everything a real run does happens here — the gate, the shard slice, the
backbone, the augmentation, the bank writer, the invariant check — on ~40
images. If this passes, the 8–13 hour version will too.

> The smoke bank is written under `/kaggle/temp` and is **not** a shard of the
> real bank. It is built with `--limit`, so it covers a prefix of your shard,
> and merging it into anything would corrupt the result.

In [34]:
if SMOKE:
    SMOKE_DIR = "/kaggle/temp/smoke"
    timings = {}
    for n in (8, 40):
        out = f"{SMOKE_DIR}/n{n}"
        subprocess.run(["rm", "-rf", out], check=False)
        argv = kb.run_shard_argv(
            GATE, manifest_path=MANIFEST, root=DATA_ROOT, backbone=BACKBONE,
            out_dir=out, splits=SPLITS, shard=SHARD_INDEX, n_shards=N_SHARDS,
            resume=False, workers=2, batch_size=BATCH_SIZE,
            checkpoint_every=n, limit=n)
        t0 = time.time()
        rc = kb.run_streaming(argv)
        timings[n] = time.time() - t0
        if rc != 0:
            raise SystemExit(
                f"SMOKE FAILED at n={n} (exit {rc}). Read the traceback above, "
                "then the playbook at the bottom of this notebook. Do NOT "
                "start the real run.")
        print(f"  n={n}: {timings[n]:.0f}s")

    RATE = kb.marginal_rate(8, timings[8], 40, timings[40])
    print(f"\nmeasured {RATE * 1000:.0f} ms/image (startup differenced out)")
else:
    RATE = None
    print("SMOKE is False -- skipping. Set it True if you have not run it today.")

SMOKE is False -- skipping. Set it True if you have not run it today.


In [35]:
if RATE is not None:
    sp = kb.session_plan(len(mine), RATE, checkpoint_every=CHECKPOINT_EVERY)
    print(f"shard {SHARD_INDEX}: {sp.n_images} images at {sp.seconds_per_image:.2f}"
          f" s/image = {sp.hours:.1f} h")
    print(f"  fits one session: {sp.fits_session}  (sessions needed: "
          f"{sp.sessions_needed})")
    print(f"  a session timeout would cost up to {sp.minutes_at_risk:.0f} min "
          f"of work (CHECKPOINT_EVERY={CHECKPOINT_EVERY})")
    for note in sp.notes:
        print("\n  NOTE:", note)

## 7. The real extraction

Set `SMOKE = False` in cell 0, re-run cells 0–5, then run this.

`--resume` is on. If the session dies — timeout, a browser tab closing, Kaggle
maintenance — **start a new session, re-run every cell from the top with the
same parameters, and this cell continues where it stopped.** It skips the rows
already written; you lose at most `CHECKPOINT_EVERY` images.

That works because `/kaggle/working` persists as this notebook's output, and the
`.npy` arrays are preallocated and survive a kill on their own. What a kill
actually destroys is the metadata written since the last checkpoint — hence
`CHECKPOINT_EVERY`, which costs two small parquet writes.

**Save a version before you close the tab** (Save Version → *Save & Run All* is
not what you want mid-run; use *Quick Save* to keep the output). An unsaved
session's `/kaggle/working` is not guaranteed to come back.

In [36]:
if SMOKE:
    print("SMOKE is still True. Set SMOKE = False in cell 0, re-run cells 0-5, "
          "then run this cell.")
else:
    argv = kb.run_shard_argv(
        GATE, manifest_path=MANIFEST, root=DATA_ROOT, backbone=BACKBONE,
        out_dir=OUT_DIR, splits=SPLITS, shard=SHARD_INDEX, n_shards=N_SHARDS,
        resume=True, workers=WORKERS, batch_size=BATCH_SIZE,
        checkpoint_every=CHECKPOINT_EVERY)
    print("$", " ".join(argv), "\n")
    t0 = time.time()
    rc = kb.run_streaming(argv)
    print(f"\nexit {rc} after {(time.time() - t0) / 3600:.2f} h")
    if rc != 0:
        print("\n" + "=" * 70)
        print("Paste the LAST line of the traceback above into the playbook "
              "cell at the bottom of this notebook.")

$ /usr/bin/python3 /kaggle/working/robust-aigc-detection/notebooks/run_shard.py --manifest /kaggle/input/datasets/justinbersamin/techjam-aigc-train/manifest.parquet --root /kaggle/temp/aigcdet_root --backbone siglip2l --out /kaggle/working/banks/siglip2l_shard1 --split train,val_internal --shard 1 --n-shards 5 --device cuda --batch-size 16 --checkpoint-every 200 --workers 4 --expect-manifest-sha256 7782a9ff346098d6fb065586d78828fcda909d8bbd2808bf69839c5e5da11a4c --resume 

shard 1/5: 26223 rows, row_id 26224..59446
nothing to resume from -- this is the first session

Loading weights: 100%|██████████| 792/792 [00:01<00:00, 727.47it/s, Materializing param=vision_model.post_layernorm.weight] 

extract:siglip2l: 100%|██████████| 26223/26223 [2:38:44<00:00,  2.75it/s]
shard 1 complete -> /kaggle/working/banks/siglip2l_shard1

exit 0 after 2.65 h


## 8. Hand your shard over

Publish `OUT_DIR` as a Kaggle Dataset named by the `BANK_DATASET` printed in cell 0 and
share it with whoever is doing the merge:

*Save Version → Quick Save*, wait for it to finish, then from the notebook's
Output tab choose **New Dataset**. Post the slug in the team chat with your
shard index and the row count printed below.

The merge notebook (`notebooks/kaggle_merge_train.ipynb`) attaches all five and
checks them against the frozen manifest before training anything.

In [37]:
state = kb.read_resume_state(OUT_DIR)
if not state.exists:
    print("nothing at", OUT_DIR, "-- the extraction has not run yet")
else:
    print(f"{OUT_DIR}")
    print(f"  {state.n_done}/{state.n_images} images ({state.fraction_done:.1%})")
    print(f"  backbone {state.backbone}  seed {state.seed}  "
          f"n_views {state.n_views}")
    print(f"  manifest fingerprint {str(state.manifest_sha256)[:16]}...")
    if state.n_remaining:
        print(f"\n  INCOMPLETE: {state.n_remaining} images to go. Re-run cell 7 "
              "in a fresh session with the same parameters -- do not publish a "
              "partial shard.")
    else:
        print("\n  COMPLETE. Publish it and post the slug + shard index.")
    sh(["du", "-sh", OUT_DIR])

/kaggle/working/banks/siglip2l_shard1
  26223/26223 images (100.0%)
  backbone siglip2l  seed 20260827  n_views 11
  manifest fingerprint d8c0d5108ae5be16...

  COMPLETE. Publish it and post the slug + shard index.
$ du -sh /kaggle/working/banks/siglip2l_shard1
594M	/kaggle/working/banks/siglip2l_shard1


## The 2am playbook

Paste the failing error into the cell below and it will tell you whether a
re-run can possibly help. The short version:

**Retryable — re-run the cell as-is (with `RESUME=True`, nothing is repeated):**

* `CUDA out of memory` → lower `BATCH_SIZE` to 8, then 4.
* `ReadTimeout` / `ConnectionError` → the download or the clone; just re-run.
* `MemoryError` / the kernel dies → lower `WORKERS`.

**Fatal — re-running burns an hour of a 30 h weekly budget for nothing:**

* `gated repo` / `401` / `403` → your account has not accepted the DINOv3
  licence, or `HF_TOKEN` is not attached to *this* notebook.
* `cannot resume the bank at …` → the directory holds a **different** bank; a
  parameter moved between sessions, usually `SHARD_INDEX`. Restore the
  parameters it was started with, or extract to a new `OUT_DIR`. **Do not
  delete it** — that throws away completed images.
* `verify_images: FAILED` → the attached Datasets are not what the manifest was
  frozen against. The report names which of missing / unreadable / divergent it
  is; each has a different fix. Do not extract from this copy.
* `No space left on device` → the shard was always too big. Raise `N_SHARDS`.
* `bank has no val_internal rows` → `SPLITS` was narrowed. Re-extract.
* `libcudart` / `Torch not compiled with CUDA` → something replaced torch.
  **Do not pip install torch.** Factory-reset the session (Run → Factory reset)
  and start again.

**And never, whatever the error:**

* Do not run `scripts/build_dataset.py`. The manifest is frozen; re-splitting it
  after banks exist silently misaligns labels against features.
* Do not `reset_index()` a manifest frame. It re-keys every view's RNG.
* Do not merge a `--limit`ed smoke bank into the real one.

In [38]:
# Paste the failing error (or the exception object) here.
ERROR = "CUDA out of memory. Tried to allocate 1.50 GiB"

print(kb.explain(ERROR))

[oom] RETRYABLE -- worth one re-run
Lower BATCH_SIZE (try 8, then 4) and re-run with RESUME=True; the completed images are kept. If it OOMs at batch size 4, restart the session -- something else is holding VRAM.
